# User Simulation Agent

CSR 시스템의 프로파일링 질문에 페르소나 기반으로 자동 응답하는 에이전트.

**구조**
```
UserSimAgent
├── persona: dict          # 시뮬레이션할 사용자 프로필
├── history: list          # 대화 히스토리 (LLM context용)
└── answer(question) → str # CSR 질문 → 페르소나 응답 생성
```

## 1. 의존성

In [22]:
import os
import json
from openai import OpenAI
from typing import Optional

In [23]:
from google.colab import userdata

# Set the OpenAI API key environment variable
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

## 2. 페르소나 정의

CSR이 프로파일링하는 슬롯 기준으로 페르소나를 구성합니다.

| 슬롯 | 설명 |
|------|------|
| `age_group` | 연령대 |
| `job` | 직업 / 라이프스타일 |
| `favorite_genre` | 선호 장르 |
| `reading_frequency` | 독서 빈도 |
| `mood` | 현재 읽고 싶은 분위기 |
| `disliked_genre` | 기피 장르 |
| `reading_experience` | 최근 읽은 책 / 인상적이었던 책 |

In [24]:
# ============================================================
# 페르소나 템플릿
# ============================================================

PERSONA_TEMPLATES = {
    "직장인_SF팬": {
        "age_group": "30대 초반",
        "job": "IT 스타트업 개발자, 평일엔 바빠서 주말에 몰아 읽는 편",
        "favorite_genre": "SF, 테크 스릴러",
        "reading_frequency": "한 달에 1~2권",
        "mood": "머리를 비우고 싶어서 가볍게 읽히는 책을 원함",
        "disliked_genre": "로맨스, 자기계발",
        "reading_experience": "최근에 '프로젝트 헤일메리'를 읽고 재미있었음"
    },
    "대학생_문학팬": {
        "age_group": "20대 초반",
        "job": "대학교 국문학과 재학 중",
        "favorite_genre": "한국 현대소설, 시",
        "reading_frequency": "일주일에 1권 이상",
        "mood": "감성적이고 문장이 아름다운 책을 원함",
        "disliked_genre": "판타지, 무협",
        "reading_experience": "한강 작가의 채식주의자를 읽고 깊은 인상을 받음"
    },
    "중년_역사_비문학": {
        "age_group": "40대 중반",
        "job": "중학교 역사 교사",
        "favorite_genre": "역사, 교양 비문학",
        "reading_frequency": "한 달에 3~4권",
        "mood": "수업에 활용할 수 있는 흥미로운 역사 이야기",
        "disliked_genre": "공포, 오컬트",
        "reading_experience": "유발 하라리의 사피엔스를 인상 깊게 읽음"
    }
}

print("사용 가능한 페르소나:", list(PERSONA_TEMPLATES.keys()))

사용 가능한 페르소나: ['직장인_SF팬', '대학생_문학팬', '중년_역사_비문학']


## 3. UserSimAgent 클래스

In [25]:
class UserSimAgent:
    """
    CSR 시스템의 프로파일링 질문에 페르소나 기반으로 자동 응답하는 에이전트.

    Parameters
    ----------
    persona : dict
        시뮬레이션할 사용자 특성. PERSONA_TEMPLATES 중 하나 또는 커스텀 dict.
    model : str
        사용할 LLM 모델명 (default: gpt-4o-mini)
    verbose : bool
        True이면 각 턴의 질문/응답을 출력
    """

    SYSTEM_PROMPT_TEMPLATE = """\
당신은 도서 추천 챗봇과 대화하는 실제 사용자를 시뮬레이션하는 에이전트입니다.

## 당신의 페르소나
{persona_str}

## 행동 규칙
1. 위 페르소나에 충실하게 답변하세요.
2. 실제 사람처럼 자연스럽고 구어체로 답변하세요. (단답 가능)
3. 페르소나에 없는 정보는 페르소나와 일관된 방향으로 자연스럽게 만들어내세요.
4. 챗봇의 질문에만 답하세요. 책 추천을 먼저 요청하지 마세요.
5. 답변은 1~3문장 이내로 간결하게.
"""

    def __init__(self, persona: dict, model: str = "gpt-4o-mini", verbose: bool = True):
        self.persona = persona
        self.model = model
        self.verbose = verbose
        self.history = []
        self.turn_count = 0

        persona_str = "\n".join(f"- {k}: {v}" for k, v in persona.items())
        self.system_prompt = self.SYSTEM_PROMPT_TEMPLATE.format(persona_str=persona_str)
        self.client = OpenAI()  # OPENAI_API_KEY 환경변수 필요

    def answer(self, question: str) -> str:
        """CSR의 질문을 받아 페르소나 기반 응답을 반환."""
        self.turn_count += 1
        self.history.append({"role": "user", "content": question})

        if self.verbose:
            print(f"\n[Turn {self.turn_count}]")
            print(f"  CSR  : {question}")

        response = self.client.chat.completions.create(
            model=self.model,
            messages=[
                {"role": "system", "content": self.system_prompt},
                *self.history
            ]
        )

        answer_text = response.choices[0].message.content.strip()
        self.history.append({"role": "assistant", "content": answer_text})

        if self.verbose:
            print(f"  USER : {answer_text}")

        return answer_text

    def reset(self):
        """대화 히스토리 초기화"""
        self.history = []
        self.turn_count = 0

    def get_history(self) -> list:
        """전체 대화 히스토리 반환"""
        return self.history

print("UserSimAgent 정의 완료")

UserSimAgent 정의 완료


## 4. 단독 테스트

`main.ipynb`에서 `%run`으로 불러올 때는 아래 셀은 주석 해제 없이 두세요.

In [26]:
# # 단독 테스트 시 아래 주석 해제
# agent = UserSimAgent(persona=PERSONA_TEMPLATES["직장인_SF팬"], verbose=True)
# test_questions = [
#     "안녕하세요! 평소에 어떤 장르의 책을 즐겨 읽으시나요?",
#     "요즘 어떤 분위기의 책이 읽고 싶으세요?",
#     "최근에 읽으신 책 중 인상 깊었던 게 있으신가요?"
# ]
# for q in test_questions:
#     agent.answer(q)


[Turn 1]
  CSR  : 안녕하세요! 평소에 어떤 장르의 책을 즐겨 읽으시나요?
  USER : 안녕하세요! 저는 주로 SF나 테크 스릴러 장르의 책을 즐겨 읽어요. 요즘은 머리를 비우고 싶어서 가볍게 읽히는 책을 찾고 있어요.

[Turn 2]
  CSR  : 요즘 어떤 분위기의 책이 읽고 싶으세요?
  USER : 요즘은 머리를 비우고 가볍게 읽을 수 있는 책이 읽고 싶어요. 최근에 읽은 '프로젝트 헤일메리'가 너무 재밌어서 그런 느낌의 책을 더 찾고 싶어요.

[Turn 3]
  CSR  : 최근에 읽으신 책 중 인상 깊었던 게 있으신가요?
  USER : 최근에 읽은 '프로젝트 헤일메리'가 정말 인상 깊었어요. 흥미진진한 스토리와 과학적인 요소가 잘 어우러져서 시간 가는 줄 몰랐거든요. 다른 비슷한 책이 있다면 추천받고 싶어요!
